#### Load, Clean and Prepare Data

In [48]:
import pandas as pd

# load dataset
df = pd.read_csv("../data/coffee_shop_sales.csv")

# display information
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Shape: (149116, 11)

Columns:
['Transaction ID', 'Transaction Date', 'Transaction Time', 'Transaction Quantity', 'Store ID', 'Store Location', 'Product ID', 'Unit Price', 'Product Category', 'Product Type', 'Product Detail']


In [49]:
# count missing values
missing_values = df.isnull().sum()

print("missing values:", missing_values)

# chow only columns that contain missing values
print("\ncolumns with missing values:")
print(missing_values[missing_values > 0])

missing values: Transaction ID          0
Transaction Date        0
Transaction Time        0
Transaction Quantity    0
Store ID                0
Store Location          0
Product ID              0
Unit Price              0
Product Category        0
Product Type            0
Product Detail          0
dtype: int64

columns with missing values:
Series([], dtype: int64)


In [50]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_summary

,missing_count,missing_percent
Transaction ID,0,0.0
Transaction Date,0,0.0
Transaction Time,0,0.0
Transaction Quantity,0,0.0
Store ID,0,0.0
Store Location,0,0.0
Product ID,0,0.0
Unit Price,0,0.0
Product Category,0,0.0
Product Type,0,0.0


In [51]:
print(df.dtypes)

Transaction ID            int64
Transaction Date            str
Transaction Time            str
Transaction Quantity      int64
Store ID                  int64
Store Location              str
Product ID                int64
Unit Price              float64
Product Category            str
Product Type                str
Product Detail              str
dtype: object


In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Transaction ID        149116 non-null  int64  
 1   Transaction Date      149116 non-null  str    
 2   Transaction Time      149116 non-null  str    
 3   Transaction Quantity  149116 non-null  int64  
 4   Store ID              149116 non-null  int64  
 5   Store Location        149116 non-null  str    
 6   Product ID            149116 non-null  int64  
 7   Unit Price            149116 non-null  float64
 8   Product Category      149116 non-null  str    
 9   Product Type          149116 non-null  str    
 10  Product Detail        149116 non-null  str    
dtypes: float64(1), int64(4), str(6)
memory usage: 21.8 MB


In [53]:
df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

print("\nTransaction Date:", df["Transaction Date"].dtype)


Transaction Date: datetime64[us]


/var/folders/53/vr8gs__x60sg207tybh25bv80000gn/T/ipykernel_31240/3106514235.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Transaction Date"] = pd.to_datetime(


In [54]:
# check is any dates failed conversion
invalid_dates = df["Transaction Date"].isna().sum()

print("invalid dates:", invalid_dates)

invalid dates: 0


In [55]:
duplicate_count = df.duplicated().sum()

print("duplicates:", duplicate_count)

duplicates: 398


In [56]:
# inspect duplicates
duplicates = df[df.duplicated(keep=False)]

# duplicates.head(20)

##### note on duplicate rows
Duplicate-looking transactions were retained because each record has a unique Transaction ID. Identical transaction details do not necessarily indicate duplicate records and may represent separate sales occurring at the same time. Quantities are aggregated during analysis rather than altering the original transaction-level data.

In [57]:
categorical_columns = [
    "Store Location",
    "Product Category",
    "Product Type",
    "Product Detail"
]

for column in categorical_columns:
    print(f"\n{column}:", df[column].nunique(), "unique values:")
    print(df[column].unique())


Store Location: 3 unique values:
<ArrowStringArray>
['Lower Manhattan', 'Hell's Kitchen', 'Astoria']
Length: 3, dtype: str

Product Category: 9 unique values:
<ArrowStringArray>
[            'Coffee',                'Tea', 'Drinking Chocolate',
             'Bakery',           'Flavours',          'Loose Tea',
       'Coffee beans', 'Packaged Chocolate',            'Branded']
Length: 9, dtype: str

Product Type: 29 unique values:
<ArrowStringArray>
['Gourmet brewed coffee',       'Brewed Chai tea',         'Hot chocolate',
           'Drip coffee',                 'Scone',      'Barista Espresso',
      'Brewed Black tea',      'Brewed Green tea',     'Brewed herbal tea',
              'Biscotti',                'Pastry', 'Organic brewed coffee',
 'Premium brewed coffee',         'Regular syrup',            'Herbal tea',
         'Gourmet Beans',         'Organic Beans',      'Sugar free syrup',
    'Drinking Chocolate',         'Premium Beans',              'Chai tea',
           'Gr

In [58]:
for column in categorical_columns:
    spaces = df[column].astype(str).str.strip().ne(df[column].astype(str)).sum()
    
    print(f"{column}: {spaces} values with leading/trailing spaces")

Store Location: 0 values with leading/trailing spaces
Product Category: 0 values with leading/trailing spaces
Product Type: 0 values with leading/trailing spaces
Product Detail: 1952 values with leading/trailing spaces


In [59]:
numeric_columns = [
    "Transaction ID",
    "Transaction Quantity",
    "Store ID",
    "Product ID",
    "Unit Price"
]

for column in numeric_columns:
    converted = pd.to_numeric(df[column], errors="coerce")
    invalid = converted.isna().sum()

    print(f"{column}: {invalid} non-numeric values")

Transaction ID: 0 non-numeric values
Transaction Quantity: 0 non-numeric values
Store ID: 0 non-numeric values
Product ID: 0 non-numeric values
Unit Price: 0 non-numeric values


In [60]:
print("Transaction Quantity:")
print(df["Transaction Quantity"].describe())

print("\nUnit Price:")
print(df["Unit Price"].describe())

Transaction Quantity:
count    149116.000000
mean          1.438276
std           0.542509
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           8.000000
Name: Transaction Quantity, dtype: float64

Unit Price:
count    149116.000000
mean          3.382219
std           2.658723
min           0.800000
25%           2.500000
50%           3.000000
75%           3.750000
max          45.000000
Name: Unit Price, dtype: float64


In [61]:
data_quality_summary = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df) * 100).round(2),
    "unique_values": df.nunique(),
    "duplicate_values": df.duplicated().sum()
})

data_quality_summary

,data_type,missing_count,missing_percent,unique_values,duplicate_values
Transaction ID,int64,0,0.0,116129,398
Transaction Date,datetime64[us],0,0.0,181,398
Transaction Time,str,0,0.0,25762,398
Transaction Quantity,int64,0,0.0,6,398
Store ID,int64,0,0.0,3,398
Store Location,str,0,0.0,3,398
Product ID,int64,0,0.0,80,398
Unit Price,float64,0,0.0,41,398
Product Category,str,0,0.0,9,398
Product Type,str,0,0.0,29,398
